# Class-imbalance experiment (Colab)

This notebook works directly in the existing Drive project so checkpoints survive a Colab disconnect. Before opening it, upload `colab_imbalance_code.zip` into the project folder shown below. The first code cell copies only the three code members into the project; it does not extract data, models, or result files.

Run cells in order. The smoke cell uses a small sample. The full run uses three seeds to keep the T4 run manageable; rerun its cell after a time-budget stop to resume from Drive checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from zipfile import ZipFile
import hashlib, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/AcademicEarlyIntervention/workspace/Academic-Early-Intervention-Agent')
CODE_BUNDLE = PROJECT_DIR / 'colab_imbalance_code.zip'
assert PROJECT_DIR.is_dir(), f'Project folder not found: {PROJECT_DIR}'
assert CODE_BUNDLE.is_file(), f'Upload colab_imbalance_code.zip into {PROJECT_DIR} first'
allowed = {'requirements-extras.txt', 'run_imbalance_experiment.py', 'notebooks/colab_imbalance.ipynb'}
with ZipFile(CODE_BUNDLE) as archive:
    members = set(archive.namelist())
    if members != allowed:
        raise ValueError(f'Unexpected bundle contents: {sorted(members)}')
    for member in sorted(allowed):
        target = PROJECT_DIR / member
        target.parent.mkdir(parents=True, exist_ok=True)
        payload = archive.read(member)
        if target.exists() and target.read_bytes() == payload:
            print('Already current:', member)
            continue
        if target.exists() and member != 'requirements-extras.txt':
            raise FileExistsError(f'Refusing to replace differing code file: {target}')
        if target.exists():
            backup = target.with_name('requirements-extras.pre-imbalance.bak')
            if not backup.exists():
                shutil.copy2(target, backup)
        target.write_bytes(payload)
        print('Copied:', member)

assert (PROJECT_DIR / 'data/raw/Student Performance and Behaviour.csv').is_file()
assert (PROJECT_DIR / 'results/tuning_catboost.json').is_file()
assert (PROJECT_DIR / 'models/pipeline/catboost.joblib').is_file()
print('Project code and required data/model artifacts are present.')

## Install dependencies

The project requirements pin scikit-learn 1.7.1 to match the saved pipeline. imbalanced-learn is installed explicitly from the optional extras.

In [ ]:
%cd /content/drive/MyDrive/AcademicEarlyIntervention/workspace/Academic-Early-Intervention-Agent
%pip install -r requirements.txt 'imbalanced-learn>=0.14'

In [ ]:
import subprocess, torch, sklearn, imblearn, catboost
print('Python packages:', 'sklearn', sklearn.__version__, '| imbalanced-learn', imblearn.__version__, '| CatBoost', catboost.__version__, '| torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU runtime in Runtime > Change runtime type, then rerun this cell.')
subprocess.run(['nvidia-smi'], check=False)

## Smoke test

This is a one-seed, two-fold, 6,000-row check with reduced CatBoost iterations. It verifies that resampling affects training rows only. Its files are isolated under `results/imbalance/smoke/` and are not included in the final export.

In [ ]:
%cd /content/drive/MyDrive/AcademicEarlyIntervention/workspace/Academic-Early-Intervention-Agent
!python run_imbalance_experiment.py --device cuda --smoke --time-budget 10

## Full run (three seeds)

The 12-model results and paper files are never inputs to this output step. The experiment writes to the new `results/imbalance/` checkpoint namespace, then creates only `results/imbalance_experiment.json` and `figures/imbalance_comparison.png`. Set a smaller or larger time budget as needed. If a cell finishes with no final aggregate, rerun this cell; completed folds and seeds are reused.

In [ ]:
import subprocess, sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/AcademicEarlyIntervention/workspace/Academic-Early-Intervention-Agent')
TIME_BUDGET_MINUTES = 120
SEEDS = [0, 1, 2]  # three seeds; five may be substituted if runtime allows
aggregate = PROJECT_DIR / 'results/imbalance_experiment.json'
figure = PROJECT_DIR / 'figures/imbalance_comparison.png'
if aggregate.is_file() and figure.is_file():
    print('Full outputs already exist; leaving them unchanged.')
else:
    command = [sys.executable, 'run_imbalance_experiment.py', '--device', 'cuda', '--time-budget', str(TIME_BUDGET_MINUTES), '--seeds', *map(str, SEEDS)]
    while True:
        completed = subprocess.run(command, cwd=PROJECT_DIR)
        if completed.returncode != 0:
            raise RuntimeError(f'Experiment command failed: exit code {completed.returncode}')
        if aggregate.is_file() and figure.is_file():
            print('Full run complete.')
            break
        print('Time budget ended with checkpoints saved. Rerun this cell to resume.')
        break

## Export only the new result files

The zip contains the aggregate JSON, comparison figure, and full-run JSON checkpoints/manifest. It excludes code, data, model binaries, smoke artifacts, the existing 12-model results, and the paper. It is saved into the project folder on Drive and downloaded to this browser session.

In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED

PROJECT_DIR = Path('/content/drive/MyDrive/AcademicEarlyIntervention/workspace/Academic-Early-Intervention-Agent')
aggregate = PROJECT_DIR / 'results/imbalance_experiment.json'
figure = PROJECT_DIR / 'figures/imbalance_comparison.png'
if not aggregate.is_file() or not figure.is_file():
    raise FileNotFoundError('Run the full experiment to completion before exporting.')
export_path = PROJECT_DIR / 'imbalance_results_export.zip'
if export_path.exists():
    raise FileExistsError(f'Refusing to overwrite {export_path}; rename the old export first.')
members = [aggregate, figure]
checkpoint_root = PROJECT_DIR / 'results/imbalance/checkpoints'
if checkpoint_root.exists():
    members += sorted(p for p in checkpoint_root.rglob('*') if p.is_file() and not p.name.endswith('.tmp'))
with ZipFile(export_path, 'w', compression=ZIP_DEFLATED) as archive:
    for member in members:
        archive.write(member, member.relative_to(PROJECT_DIR).as_posix())
print('Exported only new result files:', export_path)
files.download(str(export_path))